# Lab 02 — 乗車データの取り込みと加工

`Files/raw` の Parquet を読み込み、クレンジングと派生列の追加を行って Delta テーブル `green_tripdata_2026` に保存します。

- 元ファイルは変更しません。書き込むのは `green_tripdata_2026` だけです。
- 重複排除や、乗客数が欠損した行以外の行削除はしません。
- `trip_id` は欠損のない 8 列で並べた連番です。再実行しても同じ値になります。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
SOURCE = "Files/raw/green_tripdata_2026-04.parquet"
TABLE = "green_tripdata_2026"

raw = spark.read.parquet(SOURCE)
source_rows, source_columns = raw.count(), len(raw.columns)
print(f"Source: {source_rows:,} rows / {source_columns} columns")

## 加工ルール

| 列 | ルール |
|---|---|
| `ehail_fee` | 削除 |
| `passenger_count` | null の行だけを除外 |
| 日時 2 列 | タイムゾーンを持たない値を UTC の timestamp として保存（SQL エンドポイントで日時を扱えるようにする） |
| `pickup_date` / `pickup_hour` / `pickup_month` | 乗車日時から日付・0〜23 時・`yyyy-MM` |
| `trip_duration_minutes` | 降車 − 乗車（分、小数 2 桁）。欠損または負の値は null |
| `fare_per_mile` | `fare_amount / trip_distance`（小数 2 桁）。距離が 0 以下・欠損なら null |
| `passenger_group` | 1 人 = `single`、2 人以上 = `group`、0 以下 = `unknown` |
| `trip_id` | 乗車・降車日時、VendorID、乗降地、距離、運賃、合計金額の順に並べた連番 |

In [ ]:
ORDER_COLUMNS = ["lpep_pickup_datetime", "lpep_dropoff_datetime", "VendorID", "PULocationID",
                 "DOLocationID", "trip_distance", "fare_amount", "total_amount"]

df = (
    raw.drop("ehail_fee")
    .filter(F.col("passenger_count").isNotNull())
    .withColumn("lpep_pickup_datetime", F.col("lpep_pickup_datetime").cast("timestamp"))
    .withColumn("lpep_dropoff_datetime", F.col("lpep_dropoff_datetime").cast("timestamp"))
)
null_order_values = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ORDER_COLUMNS]).first().asDict()
if any(null_order_values.values()):
    raise ValueError(f"trip_id の並び替え列に欠損があります: {null_order_values}")

seconds = F.col("lpep_dropoff_datetime").cast("long") - F.col("lpep_pickup_datetime").cast("long")
df = (
    df.withColumn("trip_id", F.row_number().over(Window.orderBy(*ORDER_COLUMNS)).cast("long"))
    .withColumn("pickup_date", F.to_date("lpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("lpep_pickup_datetime"))
    .withColumn("pickup_month", F.date_format("lpep_pickup_datetime", "yyyy-MM"))
    .withColumn("trip_duration_minutes", F.when(seconds >= 0, F.round(seconds / 60.0, 2)))
    .withColumn("fare_per_mile", F.when((F.col("trip_distance") > 0) & F.col("fare_amount").isNotNull(),
                                        F.round(F.col("fare_amount") / F.col("trip_distance"), 2)))
    .withColumn("passenger_group", F.when(F.col("passenger_count") == 1, "single")
                .when(F.col("passenger_count") >= 2, "group").otherwise("unknown"))
)
df = df.select("trip_id", *[c for c in df.columns if c != "trip_id"])

In [ ]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)

saved = spark.table(TABLE)
saved_rows = saved.count()
save_result("nb_02_prepare_trips", {
    "source": {"path": SOURCE, "rows": source_rows, "columns": source_columns},
    "removedNullPassengerCount": source_rows - saved_rows,
    "droppedColumns": ["ehail_fee"],
    "addedColumns": ["trip_id", "pickup_date", "pickup_hour", "pickup_month", "trip_duration_minutes", "fare_per_mile", "passenger_group"],
    "saved": {"table": TABLE, "rows": saved_rows, "columns": len(saved.columns)},
    "next": "答え合わせは nb_02_validate_trips で実行します。",
})